<a href="https://colab.research.google.com/github/a-mhamdi/jlai/blob/main/Codes/Julia/Part-3/transfer-learning/transfer-learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transfer Learning
---

In [ ]:
versioninfo()  # -> 1.12.6

In [ ]:
pkgs = """[deps]
CUDA = "052768ef-5323-5732-b1bb-66c8b64840ba"
Flux = "587475ba-b771-5e3f-ad9e-33799f191a9c"
MLDatasets = "eb30cadb-4394-5ae3-aed4-317e484a6458"
Metalhead = "dbeba491-748d-5e0e-a39e-b530a07fa0cc"
ProgressMeter = "92933f4c-e287-5a05-a399-4b506db050ca"
Statistics = "10745b16-79ce-11e8-11f9-7d13ad32a3b2"
cuDNN = "02a925ec-e4fe-4b08-9a7e-0d78e3d38ccd"
"""

open("Project.toml", "w") do file
    write(file, pkgs)
end

In [ ]:
_ = begin
  import Pkg;
  Pkg.activate(".");
  Pkg.instantiate();
end

In [ ]:
Pkg.status()

Load the pre-trained model

In [ ]:
using Metalhead

In [ ]:
resnet = ResNet(18; pretrain=true)

[API Reference](https://fluxml.ai/Metalhead.jl/dev/api/reference/#API-Reference)

In [ ]:
using Flux
using Flux: onecold, onehotbatch

In [ ]:
using CUDA

In [ ]:
model = Chain(
    resnet.layers[1:end-1],
    resnet.layers[end][1:end-1],
    # Replace the last layer
    Dense(512, 10)
    ) |> gpu

Load the MNIST dataset

In [ ]:
using MLDatasets

In [ ]:
function get_data(split; batchsize=64)
    d = MNIST(split)
    X = reshape(d.features, 28, 28, 1, :)
    y = onehotbatch(d.targets, 0:9)
    loader = Flux.DataLoader((X, y); batchsize, shuffle=(split == :train))
    return loader
end

In [ ]:
train_loader = get_data(:train);
test_loader = get_data(:test);

**Define a loss function and an optimizer**

In [ ]:
loss_fn = Flux.logitcrossentropy
opt_state = Flux.setup(Adam(3e-3), model[end]) # Freeze the weights of the pre-trained layers

In [ ]:
# Per-batch preprocessing
const IMAGENET_MEAN = reshape(Float32[0.485, 0.456, 0.406], 1, 1, 3, 1)
const IMAGENET_STD  = reshape(Float32[0.229, 0.224, 0.225], 1, 1, 3, 1)

In [ ]:
function to_resnet_input(X; size=(64, 64))
    X = repeat(X; outer=(1, 1, 3, 1))                # 28×28×1×B -> 28×28×3×B
    X = Flux.upsample_bilinear(X; size=size)         # 28×28 -> 64×64 (or 224×224)
    return (X .- IMAGENET_MEAN) ./ IMAGENET_STD      # ImageNet normalization
end

Fine-tune the model

In [ ]:
using ProgressMeter, Statistics

In [ ]:
for epoch in 1:5
    @info "TRAINING..."
    Flux.trainmode!(model)
    p = Progress(length(train_loader); desc="Epoch $epoch train: ", showspeed=true)

    for (X, y) in train_loader
        X = X |> to_resnet_input
        X, y = (X, y) |> gpu

        local ŷ
        loss, ∇ = Flux.withgradient(model) do m
            ŷ = m(X)
            loss_fn(ŷ, y)
        end
        Flux.update!(opt_state, model.layers[end], ∇[1].layers[end])

        acc = mean(Flux.onecold(ŷ) .== Flux.onecold(y))
        next!(p; showvalues = [
            (:train_loss, round(loss; digits=4)),
            (:train_acc,  round(100 * acc; digits=2)),
        ])
    end
    finish!(p)

    @info "TESTING..."
    Flux.testmode!(model)
    p = Progress(length(test_loader); desc="Epoch $epoch test:  ", showspeed=true)
    tot_loss, correct, total = 0f0, 0, 0

    for (X, y) in test_loader
        X = X |> to_resnet_input
        X, y = (X, y) |> gpu
        ŷ = model(X)

        n = size(y, 2)
        tot_loss += loss_fn(ŷ, y) * n
        correct  += sum(Flux.onecold(ŷ) .== Flux.onecold(y))
        total    += n

        next!(p; showvalues = [
            (:test_loss, round(tot_loss / total; digits=4)),
            (:test_acc,  round(100 * correct / total; digits=2)),
        ])
    end
    finish!(p)
end